# Cálculo de Métricas del Proyecto
## Funeraria Aranzabal - Inventario Inteligente

Este notebook calcula todas las métricas requeridas para el proyecto.
Las métricas manuales están claramente etiquetadas con **[MANUAL]** en el título de la celda.

In [1]:
import os
import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter
import cv2

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================
PROJECT_ROOT = Path(r'C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente')
RAW_IMAGES_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATASET_DIR = PROCESSED_DIR / 'dataset'
RESULTADOS_DIR = PROCESSED_DIR / 'resultados'
MODELS_DIR = PROJECT_ROOT / 'notebooks' / 'models'
WEIGHTS_DIR = PROJECT_ROOT / 'models_weights'

# Archivos de datos
EXCEL_CRUDO = PROJECT_ROOT / 'Dataset_crudo.xlsx'
COMPARATIVA_XLSX = RESULTADOS_DIR / 'comparativa_modelos.xlsx'
PREDICCIONES_XLSX = RESULTADOS_DIR / 'predicciones_todos_modelos.xlsx'
METRICAS_PREPROC_XLSX = DATASET_DIR / 'metricas_preprocesamiento.xlsx'
COMPARISON_RESULTS_CSV = WEIGHTS_DIR / 'comparison_results.csv'
MODEL_METADATA_JSON = MODELS_DIR / 'model_metadata.json'

print('Rutas configuradas correctamente.')
print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'Imágenes raw: {RAW_IMAGES_DIR} (existe: {RAW_IMAGES_DIR.exists()})')
print(f'Dataset dir: {DATASET_DIR} (existe: {DATASET_DIR.exists()})')
print(f'Resultados dir: {RESULTADOS_DIR} (existe: {RESULTADOS_DIR.exists()})')

Rutas configuradas correctamente.
Raíz del proyecto: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente
Imágenes raw: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\raw (existe: True)
Dataset dir: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset (existe: True)
Resultados dir: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados (existe: True)


---
# Sección 1: Digitalización y Etiquetación de Imágenes
Métricas: Completitud, Ratio de Imágenes Aptas, Tasa de Rechazo

## 1.1 Métrica de Completitud
Porcentaje de registros del cuaderno físico que fueron fotografiados y subidos correctamente.

In [2]:
# Cargar el Excel crudo para contar registros totales
df_crudo = pd.read_excel(EXCEL_CRUDO)
total_registros_excel = len(df_crudo)

# Contar imágenes en data/raw/
imagenes_raw = [f for f in os.listdir(RAW_IMAGES_DIR) if f.endswith('.png')]
total_imagenes_raw = len(imagenes_raw)

# Verificar secuencia de imágenes (0.png a N.png)
numeros_imagenes = sorted([int(f.replace('.png', '')) for f in imagenes_raw])
imagen_min = numeros_imagenes[0]
imagen_max = numeros_imagenes[-1]
esperado = set(range(imagen_min, imagen_max + 1))
existentes = set(numeros_imagenes)
faltantes = sorted(esperado - existentes)

print(f'Registros en Excel (cuaderno): {total_registros_excel}')
print(f'Imágenes en data/raw/: {total_imagenes_raw}')
print(f'Rango de imágenes: {imagen_min}.png a {imagen_max}.png')
print(f'Imágenes faltantes en secuencia: {faltantes if faltantes else "Ninguna"}')

# Completitud: imágenes válidas / registros del cuaderno
completitud = (total_imagenes_raw / total_registros_excel) * 100
print(f'\n--- RESULTADO ---')
print(f'Completitud: {completitud:.2f}% ({total_imagenes_raw}/{total_registros_excel})')

Registros en Excel (cuaderno): 340
Imágenes en data/raw/: 340
Rango de imágenes: 0.png a 340.png
Imágenes faltantes en secuencia: [197]

--- RESULTADO ---
Completitud: 100.00% (340/340)


## 1.2 Ratio de Imágenes Aptas
Porcentaje de fotos que cumplen con el brillo y nitidez necesarios para ser procesadas.

In [3]:
def analizar_calidad_imagen(ruta):
    """Analiza brillo y nitidez de una imagen."""
    try:
        img = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        # Brillo: promedio de intensidad de pixeles (0-255)
        brillo = np.mean(img)
        # Nitidez: varianza del Laplaciano (mayor = mas nitida)
        laplaciano = cv2.Laplacian(img, cv2.CV_64F)
        nitidez = laplaciano.var()
        return {'brillo': brillo, 'nitidez': nitidez}
    except Exception:
        return None

# Umbrales de calidad (ajustados para escaneos con fondo blanco)
BRILLO_MIN = 100    # Minimo brillo (escaneos oscuros = mal escaneo)
NITIDEZ_MIN = 30   # Minima nitidez (documentos escaneados tienen baja varianza Laplaciana)

# Analizar todas las imagenes
resultados_calidad = []
for img_file in sorted(imagenes_raw, key=lambda x: int(x.replace('.png', ''))):
    ruta = RAW_IMAGES_DIR / img_file
    resultado = analizar_calidad_imagen(ruta)
    if resultado:
        # Para escaneos: rechazar si es muy oscuro o muy borroso
        apta = resultado['brillo'] >= BRILLO_MIN and resultado['nitidez'] >= NITIDEZ_MIN
        resultados_calidad.append({
            'imagen': img_file,
            'brillo': resultado['brillo'],
            'nitidez': resultado['nitidez'],
            'apta': apta
        })

df_calidad = pd.DataFrame(resultados_calidad)
total_analizadas = len(df_calidad)
total_aptas = int(df_calidad['apta'].sum())
total_rechazadas = total_analizadas - total_aptas

ratio_aptas = (total_aptas / total_analizadas) * 100
tasa_rechazo = (total_rechazadas / total_analizadas) * 100

print(f'Imagenes analizadas: {total_analizadas}')
print(f'Imagenes aptas: {total_aptas}')
print(f'Imagenes rechazadas: {total_rechazadas}')
print(f'\n--- RESULTADOS ---')
print(f'Ratio de Imagenes Aptas: {ratio_aptas:.2f}%')
print(f'Tasa de Rechazo: {tasa_rechazo:.2f}%')

# Mostrar estadisticas de calidad
print(f'\nEstadisticas de calidad:')
print(f'  Brillo promedio: {df_calidad["brillo"].mean():.1f}')
print(f'  Nitidez promedio: {df_calidad["nitidez"].mean():.1f}')
if total_rechazadas > 0:
    rechazadas = df_calidad[~df_calidad['apta']]
    print(f'\nImagenes rechazadas:')
    for _, row in rechazadas.iterrows():
        motivo = []
        if row['brillo'] < BRILLO_MIN:
            motivo.append(f"brillo bajo ({row['brillo']:.0f})")
        if row['nitidez'] < NITIDEZ_MIN:
            motivo.append(f"nitidez baja ({row['nitidez']:.0f})")
        print(f"    {row['imagen']}: {chr(44).join(motivo)}")

Imagenes analizadas: 340
Imagenes aptas: 246
Imagenes rechazadas: 94

--- RESULTADOS ---
Ratio de Imagenes Aptas: 72.35%
Tasa de Rechazo: 27.65%

Estadisticas de calidad:
  Brillo promedio: 234.1
  Nitidez promedio: 1293.9

Imagenes rechazadas:
    0.png: nitidez baja (24)
    1.png: nitidez baja (28)
    2.png: nitidez baja (23)
    3.png: nitidez baja (17)
    4.png: nitidez baja (23)
    5.png: nitidez baja (12)
    6.png: nitidez baja (16)
    7.png: nitidez baja (23)
    8.png: nitidez baja (30)
    9.png: nitidez baja (25)
    10.png: nitidez baja (19)
    11.png: nitidez baja (14)
    17.png: nitidez baja (27)
    18.png: nitidez baja (13)
    19.png: nitidez baja (23)
    20.png: nitidez baja (25)
    21.png: nitidez baja (28)
    25.png: nitidez baja (20)
    27.png: nitidez baja (21)
    28.png: nitidez baja (19)
    29.png: nitidez baja (16)
    33.png: nitidez baja (28)
    34.png: nitidez baja (21)
    35.png: nitidez baja (28)
    38.png: nitidez baja (24)
    42.png: nit

---
# Sección 2: Comparación de Modelos de Reconocimiento de Escritura a Mano
Métricas: CER, WER, Latencia de Inferencia (4+ modelos)

In [4]:
# Cargar resultados del notebook 02_ext_comp_modelos.ipynb
# Estos resultados fueron calculados en el notebook de comparación
# y contienen CER, WER, Accuracy y Latencia de 4 modelos pretrained

# Resultados del notebook 02 (N=221 muestras, sin fine-tuning)
resultados_htr = pd.DataFrame({
    'Modelo': ['TrOCR-Large-EN', 'TrOCR-Base-ES', 'Multicentury-HTR', 'PARSeq-Multilingual'],
    'CER': [0.6879, 9.0920, 0.6271, 2.4688],
    'WER': [1.2178, 1.0909, 1.1903, 1.0000],
    'Accuracy': [0.0181, 0.0000, 0.0860, 0.0000],
    'Latencia_Media_ms': [543.90, 50785.28, 1870.74, 467.22],
    'Latencia_P95_ms': [543.90, 66702.99, 3710.20, 705.70],
    'Muestras': [221, 221, 221, 221]
})

# Convertir latencia a segundos para la métrica requerida
resultados_htr['Latencia_Segundos'] = resultados_htr['Latencia_Media_ms'] / 1000

print('=== Comparación de Modelos HTR (Pretrained) ===')
print(f'Modelos comparados: {len(resultados_htr)} (mínimo requerido: 3)\n')
print(resultados_htr[['Modelo', 'CER', 'WER', 'Accuracy', 'Latencia_Segundos']].to_string(index=False))

# Seleccionar ganador por menor CER
ganador = resultados_htr.loc[resultados_htr['CER'].idxmin()]
print(f'\nGanador por menor CER: {ganador["Modelo"]} (CER={ganador["CER"]:.4f})')

=== Comparación de Modelos HTR (Pretrained) ===
Modelos comparados: 4 (mínimo requerido: 3)

             Modelo    CER    WER  Accuracy  Latencia_Segundos
     TrOCR-Large-EN 0.6879 1.2178    0.0181            0.54390
      TrOCR-Base-ES 9.0920 1.0909    0.0000           50.78528
   Multicentury-HTR 0.6271 1.1903    0.0860            1.87074
PARSeq-Multilingual 2.4688 1.0000    0.0000            0.46722

Ganador por menor CER: Multicentury-HTR (CER=0.6271)


---
# Sección 3: Precisión del Modelo Multicentury (Fine-tuned)
Métricas: Accuracy, CER, WER con objetivos: Accuracy >90%, CER <15%, WER <20%

In [5]:
# Cargar resultados del fine-tuning (04_ext_finetuning_HTR.ipynb)
df_finetuned = pd.read_csv(COMPARISON_RESULTS_CSV)

# Limpiar nombres de columnas (quitar flechas Unicode)
df_finetuned.columns = [c.replace(' \u2193', '').replace(' \u2191', '') for c in df_finetuned.columns]

print('=== Resultados del Modelo Multicentury-HTR (Fine-tuned) ===')
print(f'Fuente: {COMPARISON_RESULTS_CSV.name}')
print()
print(df_finetuned.to_string(index=False))

# Extraer metricas
row = df_finetuned.iloc[0]
cer_val = row['CER']
wer_val = row['WER']
acc_val = row['Char Acc']  # Character Accuracy = 1 - CER
exact_match = row['Exact Match']

print()
print('=== Evaluacion contra Objetivos ===')
check_acc = chr(9989) if acc_val > 0.90 else chr(10060)
check_cer = chr(9989) if cer_val < 0.15 else chr(10060)
check_wer = chr(9989) if wer_val < 0.20 else chr(10060)
print(f'Accuracy (Char Acc):  {acc_val*100:.2f}%  (Objetivo: >90%)  ' + check_acc)
print(f'CER:                  {cer_val*100:.2f}%  (Objetivo: <15%)   ' + check_cer)
print(f'WER:                  {wer_val*100:.2f}%  (Objetivo: <20%)   ' + check_wer)
print(f'Exact Match:          {exact_match*100:.2f}%')

=== Resultados del Modelo Multicentury-HTR (Fine-tuned) ===
Fuente: comparison_results.csv

           model   CER    WER  Char Acc  Exact Match  Lat. Media (ms)  Lat. p95 (ms)  Muestras
Multicentury-HTR 0.334 0.5442     0.666       0.3182         55310.14       85209.88        44

=== Evaluacion contra Objetivos ===
Accuracy (Char Acc):  66.60%  (Objetivo: >90%)  ❌
CER:                  33.40%  (Objetivo: <15%)   ❌
WER:                  54.42%  (Objetivo: <20%)   ❌
Exact Match:          31.82%


---
# Sección 4: Integración del Motor de Extracción en la Plataforma Web
Métricas: Reducción de Tiempo, Tasa de Corrección Manual, Uptime

## 4.1 [MANUAL] Reducción de Tiempo de Digitación

**Instrucciones:**
1. Tomar con cronómetro el tiempo que toma **escribir a mano** los datos de 1 servicio en el sistema
2. Repetir 3 veces y calcular el promedio
3. Tomar con cronómetro el tiempo que toma **subir la foto** y que el sistema la procese automáticamente
4. Repetir 3 veces y calcular el promedio
5. Ingresar ambos valores en la celda siguiente

In [6]:
# ============================================================
# [MANUAL] Ingresar tiempos en segundos
# ============================================================
tiempo_manual_segundos = None      # Tiempo promedio escribiendo a mano (segundos)
tiempo_automatico_segundos = None  # Tiempo promedio subiendo foto (segundos)

# --- NO MODIFICAR DESDE AQUÍ ---
if tiempo_manual_segundos and tiempo_automatico_segundos:
    reduccion = ((tiempo_manual_segundos - tiempo_automatico_segundos) / tiempo_manual_segundos) * 100
    print(f'Tiempo manual: {tiempo_manual_segundos:.1f}s')
    print(f'Tiempo automático: {tiempo_automatico_segundos:.1f}s')
    print(f'\n--- RESULTADO ---')
    print(f'Reducción de Tiempo: {reduccion:.2f}%')
else:
    print('⚠️  Ingresar los tiempos en la celda de arriba para calcular la reducción.')

⚠️  Ingresar los tiempos en la celda de arriba para calcular la reducción.


## 4.2 [MANUAL] Tasa de Corrección Manual

**Instrucciones:**
1. Tomar 10 servicios que hayan sido procesados por la IA
2. Revisar cada campo extraído y contar cuántos tuvo que corregir el usuario
3. Total de campos posibles: 10 servicios × 16 campos = 160 campos
4. Ingresar el número de campos corregidos en la celda siguiente

In [7]:
# ============================================================
# [MANUAL] Ingresar cantidad de campos corregidos
# ============================================================
campos_corregidos = None  # Número de campos que el usuario tuvo que corregir
total_campos_evaluados = 160  # 10 servicios × 16 campos

# --- NO MODIFICAR DESDE AQUÍ ---
if campos_corregidos is not None:
    tasa_correccion = (campos_corregidos / total_campos_evaluados) * 100
    print(f'Campos corregidos: {campos_corregidos} / {total_campos_evaluados}')
    print(f'\n--- RESULTADO ---')
    print(f'Tasa de Corrección Manual: {tasa_correccion:.2f}%')
else:
    print('⚠️  Ingresar la cantidad de campos corregidos en la celda de arriba.')

⚠️  Ingresar la cantidad de campos corregidos en la celda de arriba.


## 4.3 [MANUAL] Uptime (Disponibilidad)

**Instrucciones:**
1. Usar un servicio como UptimeRobot (https://uptimerobot.com/) o similar
2. Configurar monitoreo del endpoint de la web
3. Dejar corriendo por al menos 24 horas
4. Ingresar el porcentaje de disponibilidad en la celda siguiente

In [8]:
# ============================================================
# [MANUAL] Ingresar porcentaje de uptime
# ============================================================
uptime_porcentaje = None  # Porcentaje de disponibilidad (ej: 99.5)

# --- NO MODIFICAR DESDE AQUÍ ---
if uptime_porcentaje is not None:
    print(f'--- RESULTADO ---')
    print(f'Uptime (Disponibilidad): {uptime_porcentaje:.2f}%')
else:
    print('⚠️  Ingresar el porcentaje de uptime en la celda de arriba.')

⚠️  Ingresar el porcentaje de uptime en la celda de arriba.


## 4.4 [AUTOMÁTICO] Consumo de Recursos (CPU/RAM) - Local
Mide el consumo de CPU y RAM al ejecutar una predicción localmente.

In [9]:
import psutil

def medir_recursos_prediccion():
    """Mide CPU y RAM al cargar un modelo y hacer una prediccion."""
    import joblib
    proceso = psutil.Process(os.getpid())
    
    # RAM antes
    ram_antes = proceso.memory_info().rss / (1024 * 1024)  # MB
    cpu_antes = psutil.cpu_percent(interval=None)
    
    # Cargar modelo ETS y hacer prediccion
    modelo_path = MODELS_DIR / 'ets_servicios_totales.pkl'
    
    if modelo_path.exists():
        modelo = joblib.load(modelo_path)
        
        # Prediccion de 6 meses
        start = time.time()
        prediccion = modelo.forecast(steps=6)
        tiempo_prediccion = time.time() - start
        
        # RAM despues
        ram_despues = proceso.memory_info().rss / (1024 * 1024)  # MB
        cpu_despues = psutil.cpu_percent(interval=0.1)
        
        ram_usada = ram_despues - ram_antes
        
        print('=== Consumo de Recursos (Local) ===')
        print('Modelo: ETS (servicios_totales)')
        print(f'Tiempo de prediccion: {tiempo_prediccion:.4f}s')
        print(f'RAM usada: {ram_usada:.2f} MB')
        print(f'RAM total proceso: {ram_despues:.2f} MB')
        print(f'CPU durante prediccion: {cpu_despues:.1f}%')
        print()
        print('--- RESULTADO ---')
        print(f'Consumo CPU: {cpu_despues:.1f}%')
        print(f'Consumo RAM: {ram_usada:.2f} MB')
        return cpu_despues, ram_usada
    else:
        print(f'Modelo no encontrado: {modelo_path}')
        return None, None

cpu_consumo, ram_consumo = medir_recursos_prediccion()

=== Consumo de Recursos (Local) ===
Modelo: ETS (servicios_totales)
Tiempo de prediccion: 0.0055s
RAM usada: 60.77 MB
RAM total proceso: 203.37 MB
CPU durante prediccion: 15.8%

--- RESULTADO ---
Consumo CPU: 15.8%
Consumo RAM: 60.77 MB


---
# Sección 5: Pre-procesamiento y Estructuración de Datos
Métricas: Tasa de Limpieza, Integridad de Datos, Consistencia Temporal

In [10]:
# Cargar métricas de pre-procesamiento del notebook 01_preprocesamiento.ipynb
df_metricas_proc = pd.read_excel(METRICAS_PREPROC_XLSX)

print('=== Métricas de Pre-procesamiento ===')
print(f'Fuente: {METRICAS_PREPROC_XLSX.name}\n')
print(df_metricas_proc.to_string(index=False))

# Intentar extraer métricas específicas
print(f'\n--- Extracción de Métricas ---')

# Buscar columnas relevantes
cols = df_metricas_proc.columns.tolist()
print(f'Columnas disponibles: {cols}')

=== Métricas de Pre-procesamiento ===
Fuente: metricas_preprocesamiento.xlsx

                              Métrica                              Valor
            Cambios totales aplicados                               1335
     Promedio de cambios por registro                               3.93
                  Integridad de Datos                             97.94%
                Consistencia Temporal               6 lagunas detectadas
         Registros con Fecha Imputada                                 42
       Registros con Fecha Sospechosa                                  0
                    Outliers en Monto                         19 (5.59%)
 Categorías Normalizadas Ataud_Modelo                                338
  Categorías Normalizadas Ataud_Color                                301
      Categorías Normalizadas Capilla                                314
Categorías Normalizadas Forma de pago                                340
                   Cobertura Temporal 2022-05-

In [11]:
# Parsear metricas del archivo de pre-procesamiento
# Valores calculados por el notebook 01_preprocesamiento.ipynb

# Integridad de Datos: extraer del DataFrame
idx_integridad = df_metricas_proc[df_metricas_proc.iloc[:, 0].str.contains(
'Integridad', na=False)].index
if len(idx_integridad) > 0:
    integridad_str = str(df_metricas_proc.iloc[idx_integridad[0], 1])
    integridad = float(integridad_str.replace('%', '').strip())
else:
    integridad = 0

# Consistencia Temporal: extraer lagunas detectadas
idx_consistencia = df_metricas_proc[df_metricas_proc.iloc[:, 0].str.contains(
'Consistencia', na=False)].index
if len(idx_consistencia) > 0:
    consistencia_str = str(df_metricas_proc.iloc[idx_consistencia[0], 1])
    n_lagunas = int(consistencia_str.split()[0])
else:
    n_lagunas = 0

# Tasa de Limpieza: calcular desde registros con fecha imputada + outliers
idx_imputada = df_metricas_proc[df_metricas_proc.iloc[:, 0].str.contains(
'Imputada', na=False)].index
idx_outliers = df_metricas_proc[df_metricas_proc.iloc[:, 0].str.contains(
'Outliers', na=False)].index
registros_imputados = int(str(df_metricas_proc.iloc[idx_imputada[0], 1])) if len(idx_imputada) > 0 else 0
outliers_str = str(df_metricas_proc.iloc[idx_outliers[0], 1]) if len(idx_outliers) > 0 else '0'
n_outliers = int(outliers_str.split()[0].replace(',', ''))
total_registros_proc = 340  # Total de registros en el dataset crudo
tasa_limpieza = ((registros_imputados + n_outliers) / total_registros_proc) * 100

print('=== Metricas de Pre-procesamiento (del notebook 01) ===')
print(f'Integridad de Datos: {integridad:.2f}%')
print(f'Consistencia Temporal: {n_lagunas} lagunas detectadas')
print(f'Tasa de Limpieza: {tasa_limpieza:.2f}% ({registros_imputados} imputados + {n_outliers} outliers de {total_registros_proc} registros)')
print(f'Cambios totales aplicados: {df_metricas_proc.iloc[0, 1]}')
print(f'Promedio de cambios por registro: {df_metricas_proc.iloc[1, 1]}')

=== Metricas de Pre-procesamiento (del notebook 01) ===
Integridad de Datos: 97.94%
Consistencia Temporal: 6 lagunas detectadas
Tasa de Limpieza: 17.94% (42 imputados + 19 outliers de 340 registros)
Cambios totales aplicados: 1335
Promedio de cambios por registro: 3.93


---
# Sección 6: Comparación de Modelos de Series Temporales
Métricas: MAE, RMSE, R² Score, Tiempo de Entrenamiento (5+ modelos)

In [12]:
# Cargar comparativa de modelos (incluye tiempo_entrenamiento)
df_comparativa = pd.read_excel(COMPARATIVA_XLSX)

print('=== Comparación de Modelos de Series Temporales ===')
print(f'Fuente: {COMPARATIVA_XLSX.name}')
print(f'Modelos comparados: {df_comparativa["modelo"].nunique()} (mínimo requerido: 5)\n')

# Mostrar por target
for target in df_comparativa['target'].unique():
    print(f'\n--- Target: {target} ---')
    df_target = df_comparativa[df_comparativa['target'] == target].copy()
    df_target['tiempo_entrenamiento'] = df_target['tiempo_entrenamiento'].round(2)
    print(df_target[['modelo', 'MAE', 'RMSE', 'R2', 'MAPE', 'tiempo_entrenamiento']].to_string(index=False))

=== Comparación de Modelos de Series Temporales ===
Fuente: comparativa_modelos.xlsx
Modelos comparados: 6 (mínimo requerido: 5)


--- Target: servicios_totales ---
  modelo      MAE     RMSE        R2       MAPE  tiempo_entrenamiento
  SARIMA 4.471714 6.426825 -0.428726  42.337118                  0.17
 Prophet 5.747336 6.426110 -0.428408 109.221000                  3.41
 XGBoost 4.892906 5.491611 -0.043171  58.539086                  1.34
LightGBM 4.057635 4.658839  0.249222  50.526685                  1.91
    LSTM 4.232826 5.205913  0.062546  82.943999                 17.51
     ETS 3.342867 4.649884  0.252106  34.572438                  0.14

--- Target: monto_total ---
  modelo           MAE          RMSE         R2        MAPE  tiempo_entrenamiento
  SARIMA 160447.259749 331176.557405 -11.137085 2657.260736                  0.05
 Prophet 101537.876716 129807.463793  -0.864638  970.131234                  0.35
 XGBoost  50955.605469  99520.277813  -0.096020   77.545283           

---
# Sección 7: Forecast Bias y Precisión en Estacionalidad
Métricas: MAPE, Forecast Bias (±5%), Precisión en Estacionalidad

In [13]:
# Cargar predicciones punto a punto
df_pred = pd.read_excel(PREDICCIONES_XLSX)

# Seleccionar el mejor modelo por target
# servicios_totales: ETS (menor MAPE)
# monto_total: XGBoost (menor MAPE)

modelos_seleccionados = {
    'servicios_totales': 'ETS',
    'monto_total': 'XGBoost'
}

print('=== Forecast Bias y Precisión en Estacionalidad ===\n')

resultados_forecast = []

for target, modelo in modelos_seleccionados.items():
    df_modelo = df_pred[(df_pred['target'] == target) & (df_pred['modelo'] == modelo)].copy()
    
    if len(df_modelo) == 0:
        print(f'No hay predicciones para {target} / {modelo}')
        continue
    
    reales = df_modelo['real'].values
    predichos = df_modelo['predicho'].values
    
    # MAPE general
    mape_general = np.mean(np.abs((reales - predichos) / np.where(reales == 0, 1, reales))) * 100
    
    # Forecast Bias: tendencia sistemática al sobrestock o quiebre
    forecast_bias = np.mean(predichos - reales) / np.mean(reales) * 100 if np.mean(reales) != 0 else 0
    
    # Precisión en Estacionalidad: MAPE solo en meses de pico (>percentil 75)
    umbral_pico = np.percentile(reales, 75)
    mascara_pico = reales >= umbral_pico
    if mascara_pico.sum() > 0:
        reales_pico = reales[mascara_pico]
        predichos_pico = predichos[mascara_pico]
        mape_pico = np.mean(np.abs((reales_pico - predichos_pico) / np.where(reales_pico == 0, 1, reales_pico))) * 100
    else:
        mape_pico = None
    
    print(f'--- {target} (Modelo: {modelo}) ---')
    print(f'  MAPE general: {mape_general:.2f}%')
    print(f'  Forecast Bias: {forecast_bias:+.2f}%  (Objetivo: ±5%)  {"✅" if abs(forecast_bias) <= 5 else "❌"}')
    if mape_pico is not None:
        print(f'  Precisión Estacionalidad (MAPE en picos): {mape_pico:.2f}%  (meses con demanda ≥ {umbral_pico:.0f})')
    else:
        print(f'  Precisión Estacionalidad: No hay meses de pico suficientes')
    print()
    
    resultados_forecast.append({
        'target': target,
        'modelo': modelo,
        'MAPE_general': round(mape_general, 2),
        'Forecast_Bias': round(forecast_bias, 2),
        'MAPE_estacionalidad': round(mape_pico, 2) if mape_pico else None
    })

df_forecast = pd.DataFrame(resultados_forecast)

=== Forecast Bias y Precisión en Estacionalidad ===

--- servicios_totales (Modelo: ETS) ---
  MAPE general: 34.57%
  Forecast Bias: -20.46%  (Objetivo: ±5%)  ❌
  Precisión Estacionalidad (MAPE en picos): 48.43%  (meses con demanda ≥ 14)

--- monto_total (Modelo: XGBoost) ---
  MAPE general: 77.55%
  Forecast Bias: +34.52%  (Objetivo: ±5%)  ❌
  Precisión Estacionalidad (MAPE en picos): 78.84%  (meses con demanda ≥ 45002)



---
# Sección 8: Integración del Módulo de Predicción al Sistema Web
Métricas: Tiempo de Respuesta, Tasa de Éxito de API, Consumo de Recursos

## 8.1 [MANUAL] Tiempo de Respuesta del Endpoint

**Instrucciones:**
1. Asegurarse de que el endpoint de predicciones esté corriendo (puerto 9000)
2. Ejecutar 10 peticiones al endpoint `POST /predictions/predict`
3. Medir el tiempo de cada petición con `curl` o Postman
4. Ingresar los 10 tiempos en la celda siguiente

In [14]:
# ============================================================
# [MANUAL] Ingresar tiempos de respuesta en segundos
# ============================================================
tiempos_respuesta = []  # Lista de 10 tiempos en segundos (ej: [1.2, 0.8, ...])

# --- NO MODIFICAR DESDE AQUÍ ---
if tiempos_respuesta:
    tiempo_promedio = np.mean(tiempos_respuesta)
    tiempo_min = np.min(tiempos_respuesta)
    tiempo_max = np.max(tiempos_respuesta)
    print(f'=== Tiempo de Respuesta del Endpoint ===')
    print(f'Peticiones realizadas: {len(tiempos_respuesta)}')
    print(f'Tiempo promedio: {tiempo_promedio:.3f}s')
    print(f'Tiempo mínimo: {tiempo_min:.3f}s')
    print(f'Tiempo máximo: {tiempo_max:.3f}s')
    print(f'\n--- RESULTADO ---')
    print(f'Tiempo de Respuesta promedio: {tiempo_promedio:.3f}s')
else:
    print('⚠️  Ingresar los tiempos de respuesta en la celda de arriba.')

⚠️  Ingresar los tiempos de respuesta en la celda de arriba.


## 8.2 [MANUAL] Tasa de Éxito de Consumo de API

**Instrucciones:**
1. De las 10 peticiones anteriores, contar cuántas fallaron (error 4xx o 5xx)
2. Ingresar el número de peticiones fallidas en la celda siguiente

In [15]:
# ============================================================
# [MANUAL] Ingresar número de peticiones fallidas
# ============================================================
peticiones_fallidas = None  # Número de peticiones que fallaron de las 10
total_peticiones = 10

# --- NO MODIFICAR DESDE AQUÍ ---
if peticiones_fallidas is not None:
    tasa_exito = ((total_peticiones - peticiones_fallidas) / total_peticiones) * 100
    print(f'Peticiones exitosas: {total_peticiones - peticiones_fallidas} / {total_peticiones}')
    print(f'\n--- RESULTADO ---')
    print(f'Tasa de Éxito de API: {tasa_exito:.2f}%')
else:
    print('⚠️  Ingresar el número de peticiones fallidas en la celda de arriba.')

⚠️  Ingresar el número de peticiones fallidas en la celda de arriba.


---
# Sección 9: Resumen General de Métricas

In [18]:
# Compilar todas las métricas en una tabla resumen
resumen = []

# --- Sección 1: Digitalización ---
resumen.append({
    'Categoría': '1. Digitalización',
    'Métrica': 'Completitud',
    'Valor': f'{completitud:.2f}%',
    'Tipo': 'Automática'
})
resumen.append({
    'Categoría': '1. Digitalización',
    'Métrica': 'Ratio de Imágenes Aptas',
    'Valor': f'{ratio_aptas:.2f}%',
    'Tipo': 'Automática'
})
resumen.append({
    'Categoría': '1. Digitalización',
    'Métrica': 'Tasa de Rechazo',
    'Valor': f'{tasa_rechazo:.2f}%',
    'Tipo': 'Automática'
})

# --- Sección 2: Comparación HTR ---
for _, row_htr in resultados_htr.iterrows():
    resumen.append({
        'Categoría': '2. Comparación HTR',
        'Métrica': f'CER - {row_htr["Modelo"]}',
        'Valor': f'{row_htr["CER"]:.4f}',
        'Tipo': 'Automática'
    })
    resumen.append({
        'Categoría': '2. Comparación HTR',
        'Métrica': f'WER - {row_htr["Modelo"]}',
        'Valor': f'{row_htr["WER"]:.4f}',
        'Tipo': 'Automática'
    })
    resumen.append({
        'Categoría': '2. Comparación HTR',
        'Métrica': f'Latencia (s) - {row_htr["Modelo"]}',
        'Valor': f'{row_htr["Latencia_Segundos"]:.3f}s',
        'Tipo': 'Automática'
    })

# --- Sección 3: Precisión Multicentury ---
resumen.append({
    'Categoría': '3. Precisión Multicentury',
    'Métrica': 'Accuracy (Char Acc)',
    'Valor': f'{acc_val*100:.2f}%',
    'Tipo': 'Automática'
})
resumen.append({
    'Categoría': '3. Precisión Multicentury',
    'Métrica': 'CER',
    'Valor': f'{cer_val*100:.2f}%',
    'Tipo': 'Automática'
})
resumen.append({
    'Categoría': '3. Precisión Multicentury',
    'Métrica': 'WER',
    'Valor': f'{wer_val*100:.2f}%',
    'Tipo': 'Automática'
})

# --- Sección 4: Integración Extracción ---
if tiempo_manual_segundos and tiempo_automatico_segundos:
    reduccion = ((tiempo_manual_segundos - tiempo_automatico_segundos) / tiempo_manual_segundos) * 100
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Reducción de Tiempo', 'Valor': f'{reduccion:.2f}%', 'Tipo': 'Manual'})
else:
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Reducción de Tiempo', 'Valor': 'Pendiente', 'Tipo': 'Manual'})

if campos_corregidos is not None:
    tasa_corr = (campos_corregidos / total_campos_evaluados) * 100
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Tasa de Corrección Manual', 'Valor': f'{tasa_corr:.2f}%', 'Tipo': 'Manual'})
else:
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Tasa de Corrección Manual', 'Valor': 'Pendiente', 'Tipo': 'Manual'})

if uptime_porcentaje is not None:
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Uptime', 'Valor': f'{uptime_porcentaje:.2f}%', 'Tipo': 'Manual'})
else:
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Uptime', 'Valor': 'Pendiente', 'Tipo': 'Manual'})

if cpu_consumo is not None:
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Consumo CPU', 'Valor': f'{cpu_consumo:.1f}%', 'Tipo': 'Automática'})
    resumen.append({'Categoría': '4. Integración Extracción', 'Métrica': 'Consumo RAM', 'Valor': f'{ram_consumo:.2f} MB', 'Tipo': 'Automática'})

# --- Sección 5: Pre-procesamiento ---
resumen.append({'Categoría': '5. Pre-procesamiento', 'Métrica': 'Tasa de Limpieza', 'Valor': f'{tasa_limpieza:.2f}%', 'Tipo': 'Automática'})
resumen.append({'Categoría': '5. Pre-procesamiento', 'Métrica': 'Integridad de Datos', 'Valor': f'{integridad:.2f}%', 'Tipo': 'Automática'})
resumen.append({'Categoría': '5. Pre-procesamiento', 'Métrica': 'Consistencia Temporal', 'Valor': f'{n_lagunas} lagunas', 'Tipo': 'Automática'})

# --- Sección 6: Comparación Series Temporales ---
for _, row_comp in df_comparativa.iterrows():
    resumen.append({
        'Categoría': '6. Series Temporales',
        'Métrica': f'{row_comp["modelo"]} - {row_comp["target"]}',
        'Valor': f'MAE={row_comp["MAE"]:.2f}, RMSE={row_comp["RMSE"]:.2f}, R²={row_comp["R2"]:.4f}, Tiempo={row_comp["tiempo_entrenamiento"]:.1f}s',
        'Tipo': 'Automática'
    })

# --- Sección 7: Forecast Bias ---
for _, row_fc in df_forecast.iterrows():
    resumen.append({
        'Categoría': '7. Forecast Bias',
        'Métrica': f'{row_fc["target"]} - {row_fc["modelo"]}',
        'Valor': f'MAPE={row_fc["MAPE_general"]}%, Bias={row_fc["Forecast_Bias"]:+.2f}%, Estacional={row_fc["MAPE_estacionalidad"]}%',
        'Tipo': 'Automática'
    })

# --- Sección 8: Integración Predicción ---
if tiempos_respuesta:
    resumen.append({'Categoría': '8. Integración Predicción', 'Métrica': 'Tiempo Respuesta', 'Valor': f'{np.mean(tiempos_respuesta):.3f}s', 'Tipo': 'Manual'})
else:
    resumen.append({'Categoría': '8. Integración Predicción', 'Métrica': 'Tiempo Respuesta', 'Valor': 'Pendiente', 'Tipo': 'Manual'})

if peticiones_fallidas is not None:
    tasa_exit = ((total_peticiones - peticiones_fallidas) / total_peticiones) * 100
    resumen.append({'Categoría': '8. Integración Predicción', 'Métrica': 'Tasa Éxito API', 'Valor': f'{tasa_exit:.2f}%', 'Tipo': 'Manual'})
else:
    resumen.append({'Categoría': '8. Integración Predicción', 'Métrica': 'Tasa Éxito API', 'Valor': 'Pendiente', 'Tipo': 'Manual'})

# Crear DataFrame resumen
df_resumen = pd.DataFrame(resumen)

print('=' * 80)
print('RESUMEN GENERAL DE MÉTRICAS')
print('=' * 80)
print(f'Total métricas: {len(df_resumen)}')
print(f'Automáticas: {len(df_resumen[df_resumen["Tipo"] == "Automática"])}')
print(f'Manuales: {len(df_resumen[df_resumen["Tipo"] == "Manual"])}')
print(f'Pendientes: {len(df_resumen[df_resumen["Valor"] == "Pendiente"])}')
print()

# Mostrar por categoría
for cat in df_resumen['Categoría'].unique():
    print(f'\n--- {cat} ---')
    df_cat = df_resumen[df_resumen['Categoría'] == cat]
    for _, r in df_cat.iterrows():
        status = '⏳' if r['Valor'] == 'Pendiente' else '✅'
        print(f'  {status} {r["Métrica"]}: {r["Valor"]}')

RESUMEN GENERAL DE MÉTRICAS
Total métricas: 42
Automáticas: 37
Manuales: 5
Pendientes: 5


--- 1. Digitalización ---
  ✅ Completitud: 100.00%
  ✅ Ratio de Imágenes Aptas: 72.35%
  ✅ Tasa de Rechazo: 27.65%

--- 2. Comparación HTR ---
  ✅ CER - TrOCR-Large-EN: 0.6879
  ✅ WER - TrOCR-Large-EN: 1.2178
  ✅ Latencia (s) - TrOCR-Large-EN: 0.544s
  ✅ CER - TrOCR-Base-ES: 9.0920
  ✅ WER - TrOCR-Base-ES: 1.0909
  ✅ Latencia (s) - TrOCR-Base-ES: 50.785s
  ✅ CER - Multicentury-HTR: 0.6271
  ✅ WER - Multicentury-HTR: 1.1903
  ✅ Latencia (s) - Multicentury-HTR: 1.871s
  ✅ CER - PARSeq-Multilingual: 2.4688
  ✅ WER - PARSeq-Multilingual: 1.0000
  ✅ Latencia (s) - PARSeq-Multilingual: 0.467s

--- 3. Precisión Multicentury ---
  ✅ Accuracy (Char Acc): 66.60%
  ✅ CER: 33.40%
  ✅ WER: 54.42%

--- 4. Integración Extracción ---
  ⏳ Reducción de Tiempo: Pendiente
  ⏳ Tasa de Corrección Manual: Pendiente
  ⏳ Uptime: Pendiente
  ✅ Consumo CPU: 15.8%
  ✅ Consumo RAM: 60.77 MB

--- 5. Pre-procesamiento ---
  ✅ 

In [19]:
# Exportar resumen a Excel
output_path = RESULTADOS_DIR / 'metricas_finales.xlsx'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
df_resumen.to_excel(output_path, index=False)
print(f'Resumen exportado a: {output_path}')

Resumen exportado a: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados\metricas_finales.xlsx
